# Headcount & Capacity Forecasting

## Objective
Forecast future staffing needs based on business growth targets and historical trends.

## Key Questions
1. How many people do we need to hire in each department over the next 1-3 years?
2. Are we on track to meet business growth targets?
3. What is the hiring velocity required?
4. Which departments will grow fastest?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load data
print("Loading workforce planning data...")
history = pd.read_csv('../data/headcount_history.csv')
current_workforce = pd.read_csv('../data/current_workforce.csv')
targets = pd.read_csv('../data/business_targets.csv')

print(f"Historical data: {len(history)} monthly snapshots")
print(f"Current workforce: {len(current_workforce)} employees")
print(f"Business targets: {len(targets)} quarterly targets")

## 1. Historical Headcount Trends

In [ ]:
# Convert date and calculate total headcount by month
history['snapshot_date'] = pd.to_datetime(history['snapshot_date'])
history = history.sort_values('snapshot_date')

# Total headcount over time
total_by_month = history.groupby('snapshot_date')['headcount'].sum().reset_index()

print("HISTORICAL HEADCOUNT SUMMARY")
print("="*60)
print(f"Starting headcount (24 months ago): {total_by_month.iloc[0]['headcount']}")
print(f"Current headcount: {total_by_month.iloc[-1]['headcount']}")
print(f"Total growth: {total_by_month.iloc[-1]['headcount'] - total_by_month.iloc[0]['headcount']} employees")
print(f"Growth rate: {((total_by_month.iloc[-1]['headcount'] / total_by_month.iloc[0]['headcount']) - 1) * 100:.1f}%")
print("="*60)

In [ ]:
# Visualize total headcount trend
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(total_by_month['snapshot_date'], total_by_month['headcount'], 
        marker='o', linewidth=2, markersize=4, color='#2ecc71')
ax.fill_between(total_by_month['snapshot_date'], total_by_month['headcount'], 
                alpha=0.3, color='#2ecc71')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Headcount', fontsize=12)
ax.set_title('Historical Headcount Trend (24 Months)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

# Add trend line
z = np.polyfit(range(len(total_by_month)), total_by_month['headcount'], 1)
p = np.poly1d(z)
ax.plot(total_by_month['snapshot_date'], p(range(len(total_by_month))), 
        "r--", alpha=0.8, linewidth=2, label=f'Trend: +{z[0]:.1f} per month')
ax.legend()

plt.tight_layout()
plt.show()

## 2. Headcount by Department

In [ ]:
# Current headcount by department
current_date = history['snapshot_date'].max()
current_snapshot = history[history['snapshot_date'] == current_date]
current_snapshot = current_snapshot.sort_values('headcount', ascending=False)

print("\nCURRENT HEADCOUNT BY DEPARTMENT")
print("="*60)
print(current_snapshot[['department', 'headcount']].to_string(index=False))
print("="*60)
print(f"Total: {current_snapshot['headcount'].sum()}")

In [ ]:
# Visualize current distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = plt.cm.Set3(range(len(current_snapshot)))
bars = ax1.barh(current_snapshot['department'], current_snapshot['headcount'], 
                color=colors, alpha=0.8)
ax1.set_xlabel('Headcount', fontsize=12)
ax1.set_title('Current Headcount by Department', fontsize=14, fontweight='bold')

# Add value labels
for i, (idx, row) in enumerate(current_snapshot.iterrows()):
    ax1.text(row['headcount'] + 1, i, str(row['headcount']), 
            va='center', fontweight='bold')

# Pie chart
ax2.pie(current_snapshot['headcount'], labels=current_snapshot['department'],
        autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Headcount Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Department Growth Trends

In [ ]:
# Calculate growth rate by department
first_snapshot = history[history['snapshot_date'] == history['snapshot_date'].min()]
last_snapshot = history[history['snapshot_date'] == history['snapshot_date'].max()]

growth_analysis = pd.merge(
    first_snapshot[['department', 'headcount']],
    last_snapshot[['department', 'headcount']],
    on='department',
    suffixes=('_start', '_current')
)

growth_analysis['growth'] = growth_analysis['headcount_current'] - growth_analysis['headcount_start']
growth_analysis['growth_rate'] = ((growth_analysis['headcount_current'] / growth_analysis['headcount_start']) - 1) * 100
growth_analysis = growth_analysis.sort_values('growth_rate', ascending=False)

print("\nDEPARTMENT GROWTH (24 MONTHS)")
print("="*70)
print(growth_analysis[['department', 'headcount_start', 'headcount_current', 'growth', 'growth_rate']].to_string(index=False))
print("="*70)

In [ ]:
# Visualize growth rates
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if x > 50 else '#3498db' if x > 30 else '#95a5a6' 
          for x in growth_analysis['growth_rate']]
bars = ax.barh(growth_analysis['department'], growth_analysis['growth_rate'], 
               color=colors, alpha=0.8)

ax.set_xlabel('Growth Rate (%)', fontsize=12)
ax.set_title('Department Growth Rates (24 Months)', fontsize=14, fontweight='bold')
ax.axvline(growth_analysis['growth_rate'].mean(), color='red', linestyle='--', 
           linewidth=2, label=f"Average: {growth_analysis['growth_rate'].mean():.1f}%")
ax.legend()

# Add value labels
for i, val in enumerate(growth_analysis['growth_rate']):
    ax.text(val + 1, i, f"{val:.1f}%", va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Department headcount over time (stacked area chart)
fig, ax = plt.subplots(figsize=(14, 8))

# Pivot data for stacked area
pivot_data = history.pivot(index='snapshot_date', columns='department', values='headcount')

# Sort departments by current size
dept_order = current_snapshot.sort_values('headcount', ascending=False)['department'].tolist()
pivot_data = pivot_data[dept_order]

ax.stackplot(pivot_data.index, *[pivot_data[col] for col in pivot_data.columns],
             labels=pivot_data.columns, alpha=0.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Headcount', fontsize=12)
ax.set_title('Headcount Growth by Department (Stacked)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Business Target Analysis

In [ ]:
# Summarize targets by year
target_summary = targets.groupby('year').agg({
    'target_headcount': 'max'  # Max target for the year
}).reset_index()

target_summary_dept = targets.groupby(['year', 'department']).agg({
    'target_headcount': 'max'
}).reset_index()

# Current vs 2027 target
current_total = current_snapshot['headcount'].sum()
target_2027 = targets[targets['year'] == 2027].groupby('department')['target_headcount'].max()
current_by_dept = current_snapshot.set_index('department')['headcount']

hiring_need = target_2027 - current_by_dept

print("\nHIRING PLAN FOR 2027")
print("="*60)
print(f"{'Department':<20} {'Current':<10} {'Target':<10} {'Hiring Need':<15}")
print("="*60)
for dept in hiring_need.index:
    curr = current_by_dept[dept]
    tgt = target_2027[dept]
    need = hiring_need[dept]
    print(f"{dept:<20} {curr:<10} {tgt:<10} {need:<15}")
print("="*60)
print(f"{'TOTAL':<20} {current_total:<10} {int(target_2027.sum()):<10} {int(hiring_need.sum()):<15}")
print("="*60)

In [ ]:
# Visualize hiring need
fig, ax = plt.subplots(figsize=(12, 6))

hiring_need_sorted = hiring_need.sort_values(ascending=False)
colors = ['#e74c3c' if x > 15 else '#f39c12' if x > 10 else '#3498db' 
          for x in hiring_need_sorted]

bars = ax.barh(range(len(hiring_need_sorted)), hiring_need_sorted, color=colors, alpha=0.8)
ax.set_yticks(range(len(hiring_need_sorted)))
ax.set_yticklabels(hiring_need_sorted.index)
ax.set_xlabel('Number of Hires Needed', fontsize=12)
ax.set_title('2027 Hiring Plan by Department', fontsize=14, fontweight='bold')

# Add value labels
for i, val in enumerate(hiring_need_sorted):
    ax.text(val + 0.5, i, f"{int(val)}", va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Quarterly Hiring Velocity

In [ ]:
# Calculate quarterly hiring need (assuming even distribution)
quarters_2027 = 4
quarterly_hiring = hiring_need / quarters_2027

print("\nQUARTERLY HIRING VELOCITY (2027)")
print("="*60)
print(f"Total 2027 hiring need: {int(hiring_need.sum())} employees")
print(f"Average per quarter: {int(hiring_need.sum() / quarters_2027)} employees")
print(f"Average per month: {int(hiring_need.sum() / 12)} employees")
print("\nTop 3 hiring needs:")
for i, (dept, need) in enumerate(hiring_need.sort_values(ascending=False).head(3).items(), 1):
    print(f"  {i}. {dept}: {int(need)} total (~{int(need/quarters_2027)} per quarter)")
print("="*60)

## 6. Forecast Scenarios

In [ ]:
# Create three scenarios: base, optimistic, conservative
scenarios = {
    'Base Case': target_2027,
    'Optimistic (+20%)': target_2027 * 1.2,
    'Conservative (-15%)': target_2027 * 0.85
}

scenario_summary = pd.DataFrame(scenarios)
scenario_summary['Current'] = current_by_dept
scenario_summary = scenario_summary[['Current', 'Conservative (-15%)', 'Base Case', 'Optimistic (+20%)']]

print("\nHEADCOUNT SCENARIOS FOR 2027")
print("="*80)
print(scenario_summary.round(0).astype(int))
print("="*80)
print(f"\nTotal headcount:")
for col in scenario_summary.columns:
    total = scenario_summary[col].sum()
    print(f"  {col}: {int(total)}")

In [ ]:
# Visualize scenarios
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(scenario_summary.index))
width = 0.2

bars1 = ax.bar(x - width*1.5, scenario_summary['Current'], width, label='Current', color='#95a5a6')
bars2 = ax.bar(x - width*0.5, scenario_summary['Conservative (-15%)'], width, label='Conservative', color='#e67e22')
bars3 = ax.bar(x + width*0.5, scenario_summary['Base Case'], width, label='Base Case', color='#3498db')
bars4 = ax.bar(x + width*1.5, scenario_summary['Optimistic (+20%)'], width, label='Optimistic', color='#2ecc71')

ax.set_xlabel('Department', fontsize=12)
ax.set_ylabel('Headcount', fontsize=12)
ax.set_title('2027 Headcount Scenarios by Department', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenario_summary.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Capacity Planning

In [ ]:
# Assuming average time-to-fill and monthly hiring capacity
avg_time_to_fill = 60  # days
max_monthly_hires = 12  # capacity constraint

total_2027_need = int(hiring_need.sum())
months_needed = total_2027_need / max_monthly_hires

print("\nCAPACITY PLANNING ANALYSIS")
print("="*60)
print(f"Total 2027 hiring need: {total_2027_need} employees")
print(f"Average time-to-fill: {avg_time_to_fill} days")
print(f"Estimated hiring capacity: {max_monthly_hires} hires/month")
print(f"Time to complete hiring plan: {months_needed:.1f} months")
print("\nRECOMMENDATIONS:")
if months_needed > 10:
    print("  ⚠️  CAPACITY CONSTRAINT: Hiring plan may take longer than 2027")
    print("  → Consider: Increase recruiter headcount")
    print("  → Consider: Use agencies for high-volume roles")
    print("  → Consider: Internal mobility to fill some roles")
else:
    print("  ✓ Hiring plan is achievable within 2027 timeframe")
    print(f"  → Start hiring Q1 2027 to complete by Q4")
print("="*60)

## 8. Key Recommendations

In [ ]:
print("\n" + "="*70)
print("KEY RECOMMENDATIONS")
print("="*70)

print("\n1. IMMEDIATE ACTIONS (Q2 2026)")
top_3_depts = hiring_need.sort_values(ascending=False).head(3)
for dept, need in top_3_depts.items():
    print(f"   → Open {int(need/4)} {dept} requisitions for Q3-Q4 2026")

print("\n2. 2027 HIRING ROADMAP")
print(f"   → Q1 2027: Hire {int(total_2027_need * 0.25)} employees")
print(f"   → Q2 2027: Hire {int(total_2027_need * 0.30)} employees")
print(f"   → Q3 2027: Hire {int(total_2027_need * 0.25)} employees")
print(f"   → Q4 2027: Hire {int(total_2027_need * 0.20)} employees")

print("\n3. CAPACITY PLANNING")
print(f"   → Current recruiting capacity: {max_monthly_hires} hires/month")
if months_needed > 12:
    print(f"   → RECOMMENDATION: Increase capacity to {int(total_2027_need/12)} hires/month")
print("   → Consider internal mobility for 20-30% of roles")

print("\n4. BUDGET PLANNING")
avg_recruiting_cost = 5000  # per hire
total_recruiting_budget = total_2027_need * avg_recruiting_cost
print(f"   → Estimated recruiting budget: ${total_recruiting_budget:,}")
print(f"   → Breakdown: Job boards, agencies, recruiter time, onboarding")

print("\n" + "="*70)
print("Next Analysis: See '02_skills_gap_analysis.ipynb' to identify")
print("which skills to build vs. buy for 2027 growth")
print("="*70)